# 06 — Network Analysis: Top Discriminative FC Connections

Maps the top-scoring FC features (selected by the SVM pipeline) back to their
anatomical and functional network identity using the **Power-264 atlas**.

**Input:**
- `data/processed/datasetConnectivityLabels.csv`
- `data/raw/power264CommunityAffiliation.1D`
- `data/raw/power264CommunityNames.txt`

**Output:**
- `results/figures/network_pair_barplot.png`
- `results/metrics/top_connections_mapped.csv`

---
### ⚠️ Interpretation caveat
Feature scores come from a pipeline trained on the **full dataset** (not inside CV),
so they reflect association with the label but **not** generalization performance.
Treat them as exploratory, not confirmatory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from nilearn import datasets

sns.set_theme(style="whitegrid", font_scale=1.1)
RESULTS_DIR = "../results"
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/metrics",  exist_ok=True)

## 1. Load data

In [ ]:
df = pd.read_csv("../data/processed/datasetConnectivityLabels.csv")

fc_cols = [c for c in df.columns if c.startswith("fc_")]
mask = df["diagnosis"].notna()

X = df.loc[mask, fc_cols].values
y = df.loc[mask, "diagnosis"].astype(int).values

print(f"Subjects : {X.shape[0]}")
print(f"Features : {X.shape[1]:,}")
print(f"MDD / HC : {y.sum()} / {(y==0).sum()}")

## 2. Fit pipeline on full dataset

We fit once on all data **only to extract feature importance scores**.
Classification performance was already evaluated in `05_modeling_fc.ipynb` using CV.

In [ ]:
K = 1000

pipeline = Pipeline([
    ("scaler",   StandardScaler()),
    ("selector", SelectKBest(f_classif, k=K)),
    ("svm",      SVC(kernel="rbf", C=10, gamma="scale",
                     class_weight="balanced"))
])

pipeline.fit(X, y)
print("Pipeline fitted.")

## 3. Extract top-scoring connections

In [ ]:
selector = pipeline.named_steps["selector"]
selected_mask = selector.get_support()
selected_idx  = np.where(selected_mask)[0]
selected_scores = selector.scores_[selected_mask]

# Map flat index → (roi_i, roi_j) pair
N_ROIS = 264
triu_i, triu_j = np.triu_indices(N_ROIS, k=1)

top_fc = pd.DataFrame({
    "feature_name": [fc_cols[i] for i in selected_idx],
    "feature_idx":  selected_idx,
    "score":        selected_scores,
    "roi_i":        triu_i[selected_idx],
    "roi_j":        triu_j[selected_idx],
}).sort_values("score", ascending=False)

print(f"Top {K} connections extracted.")
print(top_fc.head(10).to_string(index=False))

## 4. Load Power-264 atlas metadata

In [ ]:
# MNI coordinates from nilearn
power = datasets.fetch_coords_power_2011()
coords = power.rois.copy()
coords["roi_id"] = np.arange(len(coords))
coords = coords.rename(columns={"x": "x_mni", "y": "y_mni", "z": "z_mni"})

# Community labels from local files
community_ids = pd.read_csv(
    "../data/raw/power264CommunityAffiliation.1D",
    header=None, names=["community_id"]
)
community_names = pd.read_csv(
    "../data/raw/power264CommunityNames.txt",
    header=None, names=["network_name"]
)

coords["community_id"] = community_ids["community_id"].values
community_map = {i + 1: name for i, name in enumerate(community_names["network_name"])}
coords["network_name"] = coords["community_id"].map(community_map)

print(f"ROIs loaded: {len(coords)}")
print(f"Networks   : {coords['network_name'].nunique()}")
coords.head()

## 5. Map connections to network pairs

In [ ]:
roi_i_info = coords.add_prefix("i_")
roi_j_info = coords.add_prefix("j_")

fc_mapped = (
    top_fc
    .merge(roi_i_info, left_on="roi_i", right_on="i_roi_id", how="left")
    .merge(roi_j_info, left_on="roi_j", right_on="j_roi_id", how="left")
)

fc_mapped["connection_type"] = np.where(
    fc_mapped["i_network_name"] == fc_mapped["j_network_name"],
    "intra-network", "inter-network"
)

fc_mapped["network_pair"] = fc_mapped.apply(
    lambda r: " - ".join(sorted([str(r["i_network_name"]), str(r["j_network_name"])])),
    axis=1
)

# Save full mapped table
fc_mapped.to_csv(f"{RESULTS_DIR}/metrics/top_connections_mapped.csv", index=False)

print(fc_mapped[["feature_name", "score", "connection_type", "network_pair"]].head(10).to_string(index=False))

## 6. Network pair frequency — top discriminative connections

In [ ]:
summary_pairs = (
    fc_mapped.groupby(["network_pair", "connection_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(summary_pairs.head(20).to_string(index=False))

In [ ]:
top_pairs = summary_pairs.head(15)

palette = {"intra-network": "#4C72B0", "inter-network": "#DD8452"}

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=top_pairs,
    y="network_pair",
    x="count",
    hue="connection_type",
    palette=palette,
    ax=ax,
    dodge=False
)
ax.set_title(f"Top 15 network pairs among top-{K} discriminative FC features")
ax.set_xlabel("Number of connections")
ax.set_ylabel("")
ax.legend(title="Connection type", loc="lower right")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/network_pair_barplot.png", dpi=150)
plt.show()

## 7. Top 30 individual connections with network labels

In [ ]:
display_cols = ["feature_name", "score", "roi_i", "roi_j",
                "i_network_name", "j_network_name", "network_pair"]

fc_mapped.sort_values("score", ascending=False)[display_cols].head(30)